### Checking different columns in files

In [1]:
#pyspark jupyter notebook matplotlib pandas pyarrow

In [2]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable  # Usa el Python del entorno actual
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"  # Spark 4.2 requiere un Java compatible
os.environ["PATH"] = os.path.join(os.environ["JAVA_HOME"], "bin") + os.pathsep + os.environ["PATH"]

In [3]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.master("local[*]") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.python.worker.memory", "2g") \
    .config("spark.python.worker.reuse", "true") \
    .config("spark.python.profile", "true") \
    .config("spark.python.connect.timeout", "60s") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/14 09:30:29 WARN Utils: Your hostname, DESKTOP-EKA7S1C, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/14 09:30:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/diego/repos/data-processing/.venv/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/14 09:30:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [4]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()

In [5]:
import pyspark.pandas as ps
import pandas as pd
import re
import os
from pyspark.sql import Row
from pyspark.sql.types import *
from IPython.display import display, HTML
from pyspark.sql.functions import col, split, lit, count, when, trim, startswith, lower, to_date, datediff, avg, min, max, stddev, mean, kurtosis, skewness, lpad, expr, coalesce


/home/diego/repos/data-processing/.venv/lib/python3.12/site-packages/pyspark/pandas/__init__.py:33: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [6]:
!java -version

openjdk version "17.0.20.1" 2026-08-18
OpenJDK Runtime Environment (build 17.0.20.1+1-1-24.04-Ubuntu)
OpenJDK 64-Bit Server VM (build 17.0.20.1+1-1-24.04-Ubuntu, mixed mode, sharing)


In [7]:
spark.conf.set("spark.sql.repl.eagerEval.enabled", True) # pretty print dataframe
spark.conf.set("spark.sql.repl.eagerEval.truncate", 50) # pretty print dataframe
spark.sparkContext.setLogLevel("ERROR") # remove warnings
spark.conf.set('spark.rapids.sql.enabled','true')
spark.conf.set('spark.rapids.sql.incompatibleOps.enabled', 'true')
spark.conf.set('spark.rapids.sql.format.csv.read.enabled', 'true')
spark.conf.set('spark.rapids.sql.format.csv.enabled', 'true')

        

In [8]:
from pathlib import Path
from functools import reduce

data_dir = Path("data/processed/sivigila")
if not data_dir.is_dir():
    data_dir = Path("../data/processed/sivigila")

files = sorted(data_dir.glob("sivigila_weekly_municipal_*.parquet"))
if not files:
    raise FileNotFoundError(f"No yearly Parquet files found in {data_dir}")

sivigila_processed = reduce(
    lambda left, right: left.unionByName(right),
    (spark.read.parquet(str(path.resolve())) for path in files),
)

print(f"Loaded {len(files)} yearly files")

Loaded 19 yearly files


In [9]:
sivigila_processed.printSchema()

root
 |-- codigo_municipio: string (nullable = true)
 |-- nombre_municipio: string (nullable = true)
 |-- codigo_departamento: string (nullable = true)
 |-- ano_epidemiologico: long (nullable = true)
 |-- semana_epidemiologica: short (nullable = true)
 |-- inicio_semana: date (nullable = true)
 |-- fin_semana: date (nullable = true)
 |-- casos_total: long (nullable = true)
 |-- casos_masculinos: long (nullable = true)
 |-- casos_femeninos: long (nullable = true)
 |-- casos_hospitalizados: long (nullable = true)
 |-- casos_menor_1: long (nullable = true)
 |-- casos_1_4: long (nullable = true)
 |-- casos_5_14: long (nullable = true)
 |-- casos_15_49: long (nullable = true)
 |-- casos_50_mas: long (nullable = true)
 |-- casos_edad_desconocida: long (nullable = true)
 |-- casos_cabecera: long (nullable = true)
 |-- casos_centro_poblado: long (nullable = true)
 |-- casos_rural: byte (nullable = true)
 |-- casos_contributivo: long (nullable = true)
 |-- casos_subsidiado: long (nullable = tru

In [10]:

sivigila_processed.show(5, truncate=False)

+----------------+-------------------+-------------------+------------------+---------------------+-------------+----------+-----------+----------------+---------------+--------------------+-------------+---------+----------+-----------+------------+----------------------+--------------+--------------------+-----------+------------------+----------------+---------------+--------------+------------------+--------------+---------+------------+----------------+----------+-----------------+---------------+---------------+---------------+---------------+---------------+---------------+------------------+-----------------+---------------+-----------------+---------------+----------------+----------+-------------------------+--------------------+---------------------------+------------------------+------------------+----------------------------+-----------------------------------+------------------------------+---------------------------------+---------------------+---------------------------

In [24]:
sivigila_processed.count()

226195

## Integración Datos Poblacion

In [11]:
from pyspark.sql import functions as F

csv_path = Path("data/processed/population/population_municipal_2007_2025.csv")
if not csv_path.is_file():
    csv_path = Path("../data/processed/population/population_municipal_2007_2025.csv")
if not csv_path.is_file():
    raise FileNotFoundError(csv_path)

poblacion_csv = spark.read.option("header", True).csv(str(csv_path.resolve()))
poblacion_to_join = poblacion_csv.select(
    F.col("codigo_municipio"),
    F.col("ano_epidemiologico").cast("int").alias("ano_epidemiologico"),
    *(F.col(name).cast("long").alias(name)
      for name in poblacion_csv.columns
      if name not in {"codigo_municipio", "ano_epidemiologico"})
)
poblacion_to_join.printSchema()

root
 |-- codigo_municipio: string (nullable = true)
 |-- ano_epidemiologico: integer (nullable = true)
 |-- poblacion_total: long (nullable = true)
 |-- poblacion_hombres: long (nullable = true)
 |-- poblacion_mujeres: long (nullable = true)
 |-- poblacion_cabecera: long (nullable = true)
 |-- poblacion_rural: long (nullable = true)
 |-- poblacion_menor_1: long (nullable = true)
 |-- poblacion_1_4: long (nullable = true)
 |-- poblacion_5_14: long (nullable = true)
 |-- poblacion_15_49: long (nullable = true)
 |-- poblacion_50_mas: long (nullable = true)
 |-- poblacion_hombres_menor_1: long (nullable = true)
 |-- poblacion_hombres_1_4: long (nullable = true)
 |-- poblacion_hombres_5_14: long (nullable = true)
 |-- poblacion_hombres_15_49: long (nullable = true)
 |-- poblacion_hombres_50_mas: long (nullable = true)
 |-- poblacion_mujeres_menor_1: long (nullable = true)
 |-- poblacion_mujeres_1_4: long (nullable = true)
 |-- poblacion_mujeres_5_14: long (nullable = true)
 |-- poblacion_m

In [15]:
df_siv_pob = (
    sivigila_processed
    .join(
        poblacion_to_join,
        ["codigo_municipio", "ano_epidemiologico"],
        "left"
    )
)

In [16]:
df_siv_pob= (
    df_siv_pob

    # Total
    .withColumn(
        "incidencia_total_100k",
        F.col("casos_total") / F.col("poblacion_total") * 100000
    )

    # Área
    .withColumn(
        "incidencia_cabecera_100k",
        F.col("casos_cabecera") /
        F.col("poblacion_cabecera") * 100000
    )
    .withColumn(
        "incidencia_rural_100k",
        (
                F.col("casos_centro_poblado") +
                F.col("casos_rural")
        ) / F.col("poblacion_rural") * 100000
    )

    # Edad
    .withColumn(
        "incidencia_menor_1_100k",
        F.col("casos_menor_1") /
        F.col("poblacion_menor_1") * 100000
    )
    .withColumn(
        "incidencia_1_4_100k",
        F.col("casos_1_4") /
        F.col("poblacion_1_4") * 100000
    )
    .withColumn(
        "incidencia_5_14_100k",
        F.col("casos_5_14") /
        F.col("poblacion_5_14") * 100000
    )
    .withColumn(
        "incidencia_15_49_100k",
        F.col("casos_15_49") /
        F.col("poblacion_15_49") * 100000
    )
    .withColumn(
        "incidencia_50_mas_100k",
        F.col("casos_50_mas") /
        F.col("poblacion_50_mas") * 100000
    )
    .withColumn(
        "incidencia_masculina_100k",
        F.col("casos_masculinos") /
        F.col("poblacion_hombres") * 100000
    )
    .withColumn(
        "incidencia_femenina_100k",
        F.col("casos_femeninos") /
        F.col("poblacion_mujeres") * 100000
    )
)

In [18]:
df_siv_pob.show()

+----------------+------------------+-------------------+-------------------+---------------------+-------------+----------+-----------+----------------+---------------+--------------------+-------------+---------+----------+-----------+------------+----------------------+--------------+--------------------+-----------+------------------+----------------+---------------+--------------+------------------+--------------+---------+------------+----------------+----------+-----------------+---------------+---------------+---------------+---------------+---------------+---------------+------------------+-----------------+---------------+-----------------+---------------+----------------+----------+-------------------------+--------------------+---------------------------+------------------------+------------------+----------------------------+-----------------------------------+------------------------------+---------------------------------+---------------------+---------------------------

In [20]:
df_siv_pob.printSchema()

root
 |-- codigo_municipio: string (nullable = true)
 |-- ano_epidemiologico: long (nullable = true)
 |-- nombre_municipio: string (nullable = true)
 |-- codigo_departamento: string (nullable = true)
 |-- semana_epidemiologica: short (nullable = true)
 |-- inicio_semana: date (nullable = true)
 |-- fin_semana: date (nullable = true)
 |-- casos_total: long (nullable = true)
 |-- casos_masculinos: long (nullable = true)
 |-- casos_femeninos: long (nullable = true)
 |-- casos_hospitalizados: long (nullable = true)
 |-- casos_menor_1: long (nullable = true)
 |-- casos_1_4: long (nullable = true)
 |-- casos_5_14: long (nullable = true)
 |-- casos_15_49: long (nullable = true)
 |-- casos_50_mas: long (nullable = true)
 |-- casos_edad_desconocida: long (nullable = true)
 |-- casos_cabecera: long (nullable = true)
 |-- casos_centro_poblado: long (nullable = true)
 |-- casos_rural: byte (nullable = true)
 |-- casos_contributivo: long (nullable = true)
 |-- casos_subsidiado: long (nullable = tru

In [25]:
# 226195
df_siv_pob.count()

226195

## Integración cobertura vacunación

In [19]:
coverage_csv_path = Path("data/processed/vaccine_coverage/coberturas_municipales_2016_2025.csv")
if not coverage_csv_path.is_file():
    coverage_csv_path = Path("../data/processed/vaccine_coverage/coberturas_municipales_2016_2025.csv")
if not coverage_csv_path.is_file():
    raise FileNotFoundError(coverage_csv_path)

coberturas_vacunacion = (
    spark.read.option("header", True)
    .csv(str(coverage_csv_path.resolve()))
)
coberturas_vacunacion.printSchema()


root
 |-- CODEP: string (nullable = true)
 |-- Nombre_ET: string (nullable = true)
 |-- DIVIPOLA: string (nullable = true)
 |-- Municipio: string (nullable = true)
 |-- meta_pob_1_anio: string (nullable = true)
 |-- varicela_un_anio_dosis_unica: string (nullable = true)
 |-- %_varicela_un_anio_dosis_unica: string (nullable = true)
 |-- meta_pob_5_anios: string (nullable = true)
 |-- varicela_cinco_anios_refuerzo: string (nullable = true)
 |-- %_varicela_cinco_anios_refuerzo: string (nullable = true)
 |-- anio: string (nullable = true)



In [22]:
columnas_cobertura = [
    name for name in coberturas_vacunacion.columns
    if name not in {"CODEP", "Nombre_ET", "DIVIPOLA", "Municipio", "anio"}
]

coberturas_para_join = coberturas_vacunacion.select(
    F.col("DIVIPOLA").alias("codigo_municipio"),
    F.col("anio").cast("long").alias("ano_epidemiologico"),
    *(F.col(name).cast("double").alias(name) for name in columnas_cobertura),
)

df_siv_pob_vac = df_siv_pob.join(
    coberturas_para_join,
    ["codigo_municipio", "ano_epidemiologico"],
    "left",
)


In [23]:
df_siv_pob_vac.show()

+----------------+------------------+-------------------+-------------------+---------------------+-------------+----------+-----------+----------------+---------------+--------------------+-------------+---------+----------+-----------+------------+----------------------+--------------+--------------------+-----------+------------------+----------------+---------------+--------------+------------------+--------------+---------+------------+----------------+----------+-----------------+---------------+---------------+---------------+---------------+---------------+---------------+------------------+-----------------+---------------+-----------------+---------------+----------------+----------+-------------------------+--------------------+---------------------------+------------------------+------------------+----------------------------+-----------------------------------+------------------------------+---------------------------------+---------------------+---------------------------

In [27]:
# 226195
df_siv_pob_vac.count()

226195

In [31]:
df_siv_pob_vac.select("ano_epidemiologico").distinct().where(df_siv_pob_vac["meta_pob_1_anio"].isNull()).show()

+------------------+
|ano_epidemiologico|
+------------------+
|              2007|
|              2008|
|              2009|
|              2010|
|              2011|
|              2012|
|              2013|
|              2014|
|              2015|
+------------------+



In [32]:
df_siv_pob_vac.select("ano_epidemiologico").distinct().where(df_siv_pob_vac["meta_pob_1_anio"].isNotNull()).show()

+------------------+
|ano_epidemiologico|
+------------------+
|              2016|
|              2017|
|              2018|
|              2019|
|              2020|
|              2021|
|              2022|
|              2023|
|              2024|
|              2025|
+------------------+

